### SoRL Playground

This notebook demonstrates the SoRL post-training pipeline. Porting a OSS model, and adopt SoRL trainer to post-train the model accordingly. 

In [1]:
# Setup
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# Disable MPS for stability
if hasattr(torch.backends, 'mps'):
    torch.backends.mps.is_available = lambda: False

from transformers import TrainingArguments, AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from sorl.sorl_trainer import SorlTrainer

device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Initialize SoRL model
model_name = "Qwen/Qwen2.5-0.5B"
model = SorlModelWrapper.from_pretrained(
    model_name,
    abstract_vocab_size_list=[128],
    memory_span=1792
)
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name) 

Some weights of Qwen2ForCausalLM were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B and are newly initialized: ['lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [3]:
# ----- infer insertion mask -----
def infer_insert_mask(data, K, vocab_size, attention_mask):
    batch_size, seq_len = data.shape
    positions = torch.arange(seq_len, device=data.device).unsqueeze(0).expand(batch_size, -1)
    insert_mask = (positions % K == 0) & (positions > 0) & (attention_mask)
    return insert_mask

# ----- insert tokens / mask into data / attention mask ----- 
def insert_tokens_with_padding(input_ids, attention_mask, insert_mask, placeholder_token, pad_token_id):
    batch_size, seq_len = input_ids.shape
    
    shift = insert_mask.long().cumsum(1)
    new_positions = torch.arange(seq_len, device=input_ids.device) + shift
    
    max_new_len = new_positions[:, -1].max().item() + 1
    expanded_tokens = input_ids.new_full((batch_size, max_new_len), placeholder_token)
    expanded_tokens.scatter_(1, new_positions, input_ids)
    expanded_mask = attention_mask.new_ones(batch_size, max_new_len)
    expanded_mask.scatter_(1, new_positions, attention_mask)
    
    pad_mask = torch.arange(max_new_len, device=input_ids.device) > new_positions.max(1).values[:, None]
    expanded_tokens.masked_fill_(pad_mask, pad_token_id)
    expanded_mask.masked_fill_(pad_mask, 0)
    
    return expanded_tokens, expanded_mask
    
# ----- sorl search -----
from typing import Optional, List, Tuple, Union

def sorl_search(
    model,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    pad_token_id: int,
    n: int = 2,
    K: int = 4,
    max_iterations: int = 2,
    memory_span_abs: int = 1792,
    memory_span_traj: int = 1792,
    temperature: Union[float, torch.Tensor] = 0.0,
    truncate_seq: bool = True
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Perform SoRL search: generate rollouts and select best sequences.
    """
    
    insert_mask = infer_insert_mask(input_ids, K, model.vocab_sizes[0], attention_mask)
    expanded_data, expanded_mask = insert_tokens_with_padding(input_ids, attention_mask, insert_mask, model.vocab_sizes[0], pad_token_id)

    repeat_data = expanded_data.repeat_interleave(n, dim=0)
    repeat_mask = expanded_mask.repeat_interleave(n, dim=0)

    search_data, search_ppt = model.recursion(
        repeat_data, 
        repeat_mask,
        max_iterations=max_iterations,
        memory_span_abs=memory_span_abs,
        memory_span_traj=memory_span_traj,
        temperature=temperature
    )

    best_data, best_ppt, best_ppt_advantage = select_best_sequences(
        search_data, search_ppt, n, expanded_data.shape[0]
    )
    
    return best_data, best_ppt, best_ppt_advantage

In [5]:
from sorl.sorl_trainer import sorl_search

test_texts = [
    "Question: What is 2+2? Answer: 4",  # Long sequence
    "Question: 1+1?",  # Short sequence  
    "Hi"  # Very short
]

encoded = tokenizer(test_texts, padding=True, truncation=True, max_length=64, return_tensors="pt")
input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

batch = {"input_ids": input_ids, "attention_mask": attention_mask}
    
from sorl.sorl_trainer import select_best_sequences
# ---- sorl search ----
n = 2
K = 4
max_iterations = 2 
memory_span_abs = 512
memory_span_traj = 512
temperature = 1.0
tokens = input_ids
pad_token_id = tokenizer.pad_token_id


best_data, best_ppt, best_ppt_advantage = sorl_search(model, 
            input_ids, 
            attention_mask, 
            pad_token_id, 
            n, K, max_iterations, memory_span_abs, memory_span_traj, temperature)


In [ ]:
# We might be missing the "mask" gadget in the trainer, if we want to 
# use GSM8K alike dataset --- is there a build-in approach for it? 


# Replace your SimpleDataset with GSM8K data loading
from datasets import load_dataset
 
# Load GSM8K dataset
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")
 
# Create dataset class for GSM8K
class GSM8KDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, tokenizer, max_length=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        example = self.dataset[idx]
        
        # Format for math reasoning
        text = f"Question: {example['question']}\nAnswer: {example['answer']}"
        
        # Tokenize
        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoded["input_ids"].squeeze(),
            "attention_mask": encoded["attention_mask"].squeeze()
        }
 
# Use GSM8K dataset
train_dataset = GSM8KDataset(gsm8k_dataset, tokenizer)

# Training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./sorl_results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    warmup_steps=1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_steps=10,
    eval_steps=10,
)

# Create SoRL trainer
print("Creating SoRL trainer...")
trainer = SorlTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    num_rollouts=4,
    K=3,
    max_iterations=2,
    memory_span_abs=1792,
    memory_span_traj=1792,
    temperature=1.0,
    alpha_info_gain=10.0,
    alpha_abs=0.1,
    alpha_soft_zipf=1.0,
)

print("Trainer created successfully!")
print(f"Model vocab sizes: {model.vocab_sizes}")
print(f"Total vocab size: {model.total_vocab_size}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Creating SoRL trainer...
Trainer created successfully!
Model vocab sizes: tensor([151936,    129])
Total vocab size: 152065


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/Users/ksgk/Implementation/mod_gpt/sorl/sorl_trainer.py:258: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `SorlTrainer.__init__`. Use `processing_class` instead.
  super().__init__(


In [ ]:
# trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
wandb: Currently logged in as: fangyuan-yu18 (ksgk-hack) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/Users/ksgk/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,833.843200
2,663.091400
3,510.680100
4,384.815600
5,310.002000
6,242.054300
7,199.524200
8,170.013400
9,133.107800
10,107.045800


KeyboardInterrupt: 

In [41]:
# A few issues persist: 
# (0). the 'insert token' is wrong -- we need to "insert token", not "replace token"
# (1). Attention mask should be updated with abstract token insertion
# (2). traj_mask, abs_mask uses 'data[0]>=vocab_size' to check levels, this is wrong now
#    - so it's necessary to re-write the loss function, that's the fastest approach
